# Time Series Forecasting - Stock Market (Tesla)

## Agenda
1. Loading Time Series Data
2. Time Series Visualization
3. Time Series Analysis
4. Stationarity in Time Series
5. Time Series Differencing
6. Removing Trends and Seasonality
7. ARIMA Model
8. Seasonal ARIMA Model
9. Inferences
10. Prophet Model (Bonus)


## Problem Statement

Forecasting stock prices ahead of time can help traders and investors make better decisions. With time series data of a company's stock price over the years, one can forecast the estimated closing price for the upcoming days. Use the Tesla stock dataset to forecast the price for the upcoming days.

## Dataset Information

The dataset contains the daily stock price data for **Tesla (TSLA)**, pulled directly from Yahoo Finance.

The data contains two columns we care about - **Date** and **Close**.

## Loading Time Series Data

We will make use of the `yfinance` library to load our dataset directly (instead of a local CSV). We will follow the same steps as before to load the time series data:
1. Download the data using `yfinance`.
2. Convert the Date column to datetime, using the `to_datetime()` method from the pandas module.
3. Convert the Date column to index for easier computation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

#importing the dataset directly from Yahoo Finance
data = yf.download("TSLA", start="2015-01-01", end="2024-12-31")

#some yfinance versions return MultiIndex columns even for a single ticker - flatten them
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[['Close']]
data.columns = ['#Passengers']          # keeping the same column name style as the original hands-on

#converting the data to datetime object type
data.index = pd.to_datetime(data.index)

#displaying the data
data.head()


## Time Series Visualization

Plotting the time series can help in analyzing the visible trends, seasonality in the data. We use matplotlib to create simple line plots to study the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data.plot()
plt.show()


## Time Series Analysis

From the plot drawn in the previous section, we can look for patterns like trend, seasonality and cyclic behaviour, just like we did for the passenger dataset.

In [ ]:
mean_log = data.rolling(window=12).mean()
std_log = data.rolling(window=12).std()

plt.plot(data, color='blue', label='Original')
plt.plot(mean_log, color='red', label='Rolling Mean')
plt.plot(std_log, color='black', label='Rolling Std')
plt.legend(loc='best')
plt.title('Rolling Mean & Standard Deviation')
plt.show()


The rolling mean and standard deviation on the original time series data shows a constant increase in the mean, just like in the passenger dataset.

## Stationarity in a Time Series

Before we can begin modeling with the ARIMA model for forecasting, we have to make sure the time series data is stationary. In simple terms, if a data consists of trends and seasonality, the data is not going to be a stationary data. Stock prices almost always have a trend, so let's take a look at how we can fix this.

## How to Find the Stationarity of a Time Series?

To find the stationarity in a data, we can use the statistical test such as the Augmented Dickey Fuller test.

Here, we will take two hypotheses, a null hypotheses and an alternate hypotheses. If we are able to reject the null hypotheses after the computation, the time series is stationary series.

To reject the null hypotheses, the following must be true:
1. If the p-value after the adfuller test is greater than 0.05, we fail to reject the hypotheses.
2. If the p-value is less than 0.05, we can reject the null hypotheses and assume that the time series is stationary.

In [ ]:
#checking the stationarity of the series
from statsmodels.tsa.stattools import adfuller
result = adfuller(data['#Passengers'])
print(result)


The p-value from the result is on the index - 1, and we can check if the data is stationary or not. For stock price data, the p-value is almost always greater than 0.05 and thus, we cannot reject the null hypotheses and assume the data to be non-stationary.

## Time Series Differencing

Differencing in time series is the process of reducing the non-stationary time series to a stationary time series with a series of subtraction operations i.e. subtracting the observations from one another.

**Diff(t) = x(t) - x(t-1)**, where Diff(t) is the differenced series, x(t) is the observation at given time t, and x(t-1) is the previous observation.

## Other ways to Make Time Series Stationary

There are several ways to make the time series stationary you can choose from.
- Differencing is the most common technique to make time series stationary.
- You can use power transformation.
- Log transformations of the time series is another technique to make the time series stationary.

In [ ]:
#logarithmic transformation to make the time series stationary
first_log = np.log(data)
first_log = first_log.dropna()
first_log.plot()
plt.show()


Using the log transformation, we will try to make the time series stationary.

In [ ]:
mean_log = first_log.rolling(window=12).mean()
std_log = first_log.rolling(window=12).std()

plt.plot(first_log, color='blue', label='Original')
plt.plot(mean_log, color='red', label='Rolling Mean')
plt.plot(std_log, color='black', label='Rolling Std')
plt.legend(loc='best')
plt.title('Rolling Mean & Standard Deviation (Logarithmic Scale)')
plt.show()


From the plot, we can see that we have improved the time series a little in terms of mean and standard deviation.

## Removing Trends and Seasonality

We create a new time series, by subtracting the rolling mean with the log transformed time series.

In [ ]:
new_data = first_log - mean_log
new_data = new_data.dropna()
new_data.head()


In [ ]:
mean_log = new_data.rolling(window=12).mean()
std_log = new_data.rolling(window=12).std()

plt.plot(new_data, color='blue', label='Original')
plt.plot(mean_log, color='red', label='Rolling Mean')
plt.plot(std_log, color='black', label='Rolling Std')
plt.legend(loc='best')
plt.title('Rolling Mean & Standard Deviation (Logarithmic Scale)')
plt.show()


When we plot the rolling mean and standard deviation, we can clearly see the mean and standard deviation is bettered.

In [ ]:
#adfuller test for stationarity
result = adfuller(new_data['#Passengers'])
print(result)


The p-value from the result should now be less than 0.05, therefore we can reject the null hypotheses and consider the time series to be stationary.

In [ ]:
#seasonal Decompose
from statsmodels.tsa.seasonal import seasonal_decompose
decompose_result = seasonal_decompose(new_data['#Passengers'].dropna(), period=12)

decompose_result.plot()
plt.show()


The seasonal decomposition shows the trend to be removed, but some seasonality/cyclicality might still be present in the time series.

## Autocorrelation and Partial Autocorrelation

In [ ]:
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

acf_plot = acf(new_data['#Passengers'])
pacf_plot = pacf(new_data['#Passengers'])
plot_acf(acf_plot)
plt.show()


We use the autocorrelation plot to determine the q value for the ARIMA order.

In [ ]:
plot_pacf(pacf_plot)
plt.show()


We use the Partial autocorrelation plot to determine the p value for the ARIMA order.

## ARIMA Model For Time Series Forecasting

ARIMA model is the combination of Autoregressive(AR), Integrated (I), and Moving Average(MA) models.

Here, p, d and q are the order of AR, order of differencing and order of MA respectively. We have already calculated these values using the autocorrelation and partial autocorrelation plots, or we can deduce these values using the ACF and PACF.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

train = new_data.iloc[:-30]['#Passengers']
test = new_data.iloc[-30:]['#Passengers']

model = ARIMA(train, order=(1, 0, 2))
model_fit = model.fit()
model_fit.summary()


The ARIMA model is fit on the training data using the order - (1,0,2). We have taken the order of differencing as 0 since we are using the new transformed time series for the model.

In [ ]:
new_data['predict'] = model_fit.predict(start=len(train),
                                          end=len(train)+len(test)-1,
                                          dynamic=True)
new_data[['#Passengers', 'predict']].plot()
plt.show()


As you can see, the predictions might be way off the actual values from the test set. Therefore, we can move to the seasonal ARIMA model for our forecasting.

## SARIMA Model For Time Series Forecasting

## What is Seasonal ARIMA?

In the seasonal ARIMA model, we have to specify the seasonal order as well. The seasonal order remains the same as the ARIMA order, and we can add the periodic order in the seasonal order according to the periodicity.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX, SARIMAXResults

model = SARIMAX(train, order=(1, 0, 2), seasonal_order=(1, 0, 2, 12))
model = model.fit()

new_data['predict'] = model.predict(start=len(train),
                                     end=len(train)+len(test)-1,
                                     dynamic=True)
new_data[['#Passengers', 'predict']].plot()
plt.show()


Here, we can see the predicted values on the test set are more accurate than the ARIMA model. Therefore we have successfully created a Time series forecast model. Now we will use this model to forecast the time series.

## Inferences

We had trained the model on the transformed time series, therefore the predictions are aligned to the same. We can train the model with the original dataset, and add the order of differencing manually and get the predictions on the actual values. Or reverse transform the predictions and plot with the original series.

In [ ]:
#predicting the projections for the next few months
forecast = model.forecast(steps=60)
new_data['#Passengers'].plot()
forecast.plot()
plt.show()


## Prophet Model (Bonus)

As an addition to ARIMA and SARIMA, we can also use **Prophet** - a time series forecasting tool built and open-sourced by Meta (Facebook). It automatically handles trend and seasonality, so it needs very little setup compared to ARIMA/SARIMA.

In [ ]:
# !pip install prophet   # uncomment to install if not already installed
from prophet import Prophet

#preparing the data in the format Prophet expects: columns 'ds' and 'y'
prophet_data = data.reset_index()[['Date', '#Passengers']]
prophet_data.columns = ['ds', 'y']

#splitting into train and test, same size as before
prophet_train = prophet_data.iloc[:-30]
prophet_test = prophet_data.iloc[-30:]

#fitting the Prophet model
model = Prophet()
model.fit(prophet_train)


In [ ]:
#forecasting for the length of the test set
future = model.make_future_dataframe(periods=len(prophet_test))
forecast = model.predict(future)

model.plot(forecast)
plt.show()


In [ ]:
#plotting the trend and seasonality components
model.plot_components(forecast)
plt.show()


Just like the SARIMA model, Prophet captures the trend in the stock price and gives us a forecast for the upcoming days, but it needs much less manual tuning (no need to manually find p, d, q or check stationarity).